Group3 LSTM-CNN hybrid model, replicating og architecture
Rita Fadhel, ECE 284 Final Project, Do NOT DELETE, or change 

In [54]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder

from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [55]:
# csv + split file
CSV_PATH = "SkinBiopsyAnnotation.csv"
IMG_DIR = "images/Compressed" 

df = pd.read_csv(CSV_PATH)

df["slide"] = df["slide"].astype(str)

train_df = df[df["dataset"] == "train"].reset_index(drop=True)
test_df = df[df["dataset"] == "test"].reset_index(drop=True)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 254
Test: 73


In [56]:
#label encoding
le = LabelEncoder()
le.fit(train_df["type"])

num_classes = len(le.classes_)

print("Classes:", list(le.classes_))
print("Num classes:", num_classes)

Classes: ['AK', 'BCC', 'BCC ', 'SCC', 'SK', 'Sk', 'negative', 'nevus']
Num classes: 8


In [57]:
#dataset
class SkinDataset(Dataset):
    def __init__(self, df, img_dir, label_encoder):
        self.df = df
        self.img_dir = img_dir
        self.le = label_encoder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # robust image loading (fixes FileNotFoundError issues)
        img_path = os.path.join(self.img_dir, row["slide"] + ".jpg")

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Missing image: {img_path}")

        img = Image.open(img_path).convert("RGB")
        img = img.resize((224, 224))

        img = np.array(img).astype(np.float32) / 255.0
        img = torch.tensor(img).permute(2, 0, 1)

        label = self.le.transform([row["type"]])[0]

        return img, label

In [58]:
#dataloaders
train_dataset = SkinDataset(train_df, IMG_DIR, le)
test_dataset  = SkinDataset(test_df, IMG_DIR, le)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [59]:
#LSTM-CNN model

class LSTM_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # CNN feature extractor
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])  # (B, 512, 7, 7)

        # LSTM on spatial sequence
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=256,
            batch_first=True
        )

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)  # (B, 512, 7, 7)

        B, C, H, W = x.shape

        # convert spatial grid → sequence
        x = x.view(B, C, H * W).permute(0, 2, 1)  # (B, 49, 512)

        x, _ = self.lstm(x)

        x = x[:, -1, :]  # last timestep

        return self.fc(x)

In [60]:
model = LSTM_CNN(num_classes=num_classes).to(device)

print(model)

LSTM_CNN(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [61]:
#loss + optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 10

In [62]:
#training loop 

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10 | Loss: 1.5093
Epoch 2/10 | Loss: 1.1493
Epoch 3/10 | Loss: 0.7326
Epoch 4/10 | Loss: 0.3753
Epoch 5/10 | Loss: 0.2421
Epoch 6/10 | Loss: 0.1450
Epoch 7/10 | Loss: 0.0509
Epoch 8/10 | Loss: 0.0285
Epoch 9/10 | Loss: 0.0219
Epoch 10/10 | Loss: 0.0301


In [63]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Test Accuracy:", correct / total)

Test Accuracy: 0.5342465753424658
